<a href="https://colab.research.google.com/github/nobishun/bridge-risk-analysis/blob/main/notebooks/02_detour_calculation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02. 迂回路計算

## このノートブックの目的
各橋梁が通行止めになった場合の「迂回路長」を、OSM道路ネットワーク上の最短経路探索によって計算します。

## 入力
- `../data/interim/01_bridges_with_traffic.gpkg`（`01_data_preparation.ipynb` の出力）
- OSM道路ネットワーク（`../data/raw/tokyo_23_wards_drive.graphml`）

## 出力
- `../data/interim/02_bridges_with_detour.gpkg`
  上記に `detour_length_m`（迂回路長）・`detour_path_nodes`（迂回路のノード列）・`detour_route_wkt`（迂回路の経路ジオメトリ、ダッシュボードでの地図表示用）を追加したGeoDataFrame

## ⚠️ 実行時間について
全1,226件の迂回路計算には**約1時間**かかります（1件ごとに、その橋を含むedgeを取り除いたグラフ上で最短経路を探索するため）。そのため、このノートブックを独立したファイルとして切り出し、**計算結果をファイルに保存（キャッシュ）**しています。以降のノートブック（03, 04）はこのファイルを読み込むだけで済むため、再計算は不要です。

セクション7（迂回路ルートのジオメトリ保存）は、セクション5・6の実行結果に依存せず、保存済みの`02_bridges_with_detour.gpkg`をファイルから読み込み直す設計にしています。そのため、Colabを新しく開き直した場合でも、セクション7だけを実行すればよく、重い最短経路探索（約1時間）をやり直す必要はありません。


## 0. 環境準備

In [ ]:
!pip install osmnx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.7/104.7 kB 3.9 MB/s eta 0:00:00


In [ ]:
import json
import os

import folium
import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
from osmnx.routing import route_to_gdf
from tqdm.notebook import tqdm

## 1. 入力データの読み込み

Colabでの実行時は、フォルダ直下に01で作成した（01_bridges_with_traffic.gpkg）を保存し、INPUT_PATHは（INPUT_PATH = "/content/01_bridges_with_traffic.gpkg"）を利用してください。

In [ ]:
#INPUT_PATH = "../data/interim/01_bridges_with_traffic.gpkg"
INPUT_PATH = "/content/01_bridges_with_traffic.gpkg"
bridges_gdf = gpd.read_file(INPUT_PATH)

# WKTで保存していたOSM側のジオメトリをGeoSeriesに戻す
bridges_gdf["osm_geometry_line"] = gpd.GeoSeries.from_wkt(bridges_gdf["osm_geometry_line_wkt"])
bridges_gdf["osm_geometry_point"] = gpd.GeoSeries.from_wkt(bridges_gdf["osm_geometry_point_wkt"])

# 【ブラッシュアップ点】GPKGへの保存・再読み込みを経由すると、OSMのノードID(osm_u/osm_v)や
# edgeのkey(osm_key)がfloat型に変換されてしまうことがある。G.has_edge()はint型のノードIDを
# 前提としているため、ここで明示的にintへ変換しておく（型不一致による誤判定を防ぐため）。
for col in ["osm_u", "osm_v", "osm_key"]:
    bridges_gdf[col] = bridges_gdf[col].astype(int)

print(f"読み込んだ橋梁数: {len(bridges_gdf)}")
bridges_gdf.head()


読み込んだ橋梁数: 1226


,dpf_index,bridge_name,osm_u,osm_v,osm_key,distance_m,dpf_title,dpf_lat,dpf_lon,dpf_gyousei_kuiki_shikuchouson_mei,...,osm_junction,osm_area,index_right,voronoi_title,traffic_count_24h_auto,osm_geometry_line_wkt,osm_geometry_point_wkt,geometry,osm_geometry_line,osm_geometry_point
0,2,玉野橋,617126346,617125876,0,0.36,玉野橋,35.60737,139.64151,世田谷区,...,None,None,1424,交通調査基本区間番号:13304660040 (一般国道４６６号（第三京浜道路）),64469.0,"LINESTRING (-17390.10164 -43602.365488, -17389...",POINT (-17379.357532 -43542.859268),POINT (-17379.096 -43543.111),"LINESTRING (-17390.102 -43602.365, -17389.286 ...",POINT (-17379.358 -43542.859)
1,3,矢澤橋,537929170,537929320,0,4.80,矢澤橋,35.61337,139.64220,世田谷区,...,None,None,1389,交通調査基本区間番号:13304660020 (一般国道４６６号),48526.0,"LINESTRING (-17313.919212 -42876.502706, -1731...",POINT (-17319.905341 -42876.263124),POINT (-17315.289 -42877.587),"LINESTRING (-17313.919 -42876.503, -17317.506 ...",POINT (-17319.905 -42876.263)
2,4,無名五十七号橋,811135988,1033248469,0,4.09,無名五十七号橋,35.65581,139.63255,世田谷区,...,None,None,1386,交通調査基本区間番号:13601180090 (調布経堂停車場線),10873.0,"LINESTRING (-18189.033518 -38136.600681, -1817...",POINT (-18183.55651 -38169.310893),POINT (-18179.897 -38167.495),"LINESTRING (-18189.034 -38136.601, -18178.142 ...",POINT (-18183.557 -38169.311)
3,5,中ノ橋,563748534,822683230,0,23.75,中ノ橋,35.69250,139.67889,中野区,...,None,None,1620,交通調査基本区間番号:13403170230 (環状六号線),34486.0,"LINESTRING (-13979.72641 -34109.403722, -13975...",POINT (-13964.753522 -34084.664614),POINT (-13977.646 -34104.614),"LINESTRING (-13979.726 -34109.404, -13975.508 ...",POINT (-13964.754 -34084.665)
4,7,大井北部陸橋(ランプ部２),582140959,7093114606,0,40.83,大井北部陸橋(ランプ部２),35.59491,139.75565,品川区,...,None,None,2325,交通調査基本区間番号:13303570250 (一般国道３５７号),47937.0,"LINESTRING (-7027.818025 -44900.038352, -7059....",POINT (-7043.56124 -44898.99972),POINT (-7039.158 -44939.591),"LINESTRING (-7027.818 -44900.038, -7059.304 -4...",POINT (-7043.561 -44899)


## 2. OSM道路ネットワークの読み込み

Colabでの実行時は、フォルダ直下に01で作成した（tokyo_23_wards_drive.graphml）を保存し、INPUT_PATHは（OSM_GRAPHML_PATH = "/content/tokyo_23_wards_drive.graphml"
）を利用してください。

In [ ]:
#OSM_GRAPHML_PATH = "../data/raw/tokyo_23_wards_drive.graphml"
OSM_GRAPHML_PATH = "/content/tokyo_23_wards_drive.graphml"
if not os.path.exists(OSM_GRAPHML_PATH):
    raise FileNotFoundError(
        f"{OSM_GRAPHML_PATH} が見つかりません。"
        " 先に 01_data_preparation.ipynb を実行し、道路ネットワークを取得・保存してください。"
        " このファイルはサイズが大きいためGitHubには含めていません。"
    )

G = ox.load_graphml(filepath=OSM_GRAPHML_PATH)
print("道路ネットワークを読み込みました。")


道路ネットワークを読み込みました。


## 3. 迂回路計算ロジックの関数化

「橋のedgeを一時的にグラフから取り除き、それでも同じ地点間を結ぶ最短経路が存在するか」を探索するロジックを関数にまとめます。橋ごとに`G.copy()`でグラフを複製してから該当edgeを除去しているのは、他の橋の計算に影響を与えないようにするためです（このコピー処理のコストが、計算全体が重くなる主な要因です）。


In [ ]:
def calc_detour(G, u, v, key):
    """指定した橋(edge)を除いた場合の迂回路長と経路ノード列を計算する。

    Args:
        G: OSMの道路ネットワークグラフ(MultiDiGraph)
        u, v, key: 対象橋に対応するOSM edgeの識別子

    Returns:
        (detour_length_m, detour_path_nodes) のタプル。
        迂回路が見つからない場合は (np.inf, None)。
    """
    g_temp = G.copy()
    if not g_temp.has_edge(u, v, key):
        return np.inf, None

    g_temp.remove_edge(u, v, key)
    try:
        path_nodes = nx.shortest_path(g_temp, source=u, target=v, weight="length")
        path_nodes = [int(n) for n in path_nodes]
        edges_gdf = route_to_gdf(G, path_nodes, weight="length")
        return edges_gdf["length"].sum(), path_nodes
    except nx.NetworkXNoPath:
        return np.inf, None


## 4. 動作確認：1件だけ試す

重い処理を全件実行する前に、まず1件だけ動作確認します。

In [ ]:
sample_row = bridges_gdf.iloc[0]
sample_length, sample_path = calc_detour(G, sample_row["osm_u"], sample_row["osm_v"], sample_row["osm_key"])

if sample_length == np.inf:
    print(f"橋『{sample_row['bridge_name']}』: 迂回路が見つかりませんでした")
else:
    print(f"橋『{sample_row['bridge_name']}』: 迂回路長 = {sample_length:.1f}m")


橋『玉野橋』: 迂回路長 = 597.1m


## 5. 全橋梁で迂回路を計算（重い処理：約1,225件で1時間程度）

以下のセルは実行に時間がかかります。Colabのランタイムがタイムアウトしないよう、実行中はタブを開いたままにしておくことをおすすめします。


In [ ]:
detour_lengths = []
detour_paths = []

for _, row in tqdm(bridges_gdf.iterrows(), total=len(bridges_gdf), desc="迂回路計算"):
    length, path = calc_detour(G, row["osm_u"], row["osm_v"], row["osm_key"])
    detour_lengths.append(length)
    detour_paths.append(path)

bridges_gdf["detour_length_m"] = detour_lengths
bridges_gdf["detour_path_nodes"] = detour_paths

finite_count = (bridges_gdf["detour_length_m"] != np.inf).sum()
print(f"迂回路が見つかった橋梁数: {finite_count} / {len(bridges_gdf)}")


迂回路計算:   0%|          | 0/1226 [00:00<?, ?it/s]

迂回路が見つかった橋梁数: 1105 / 1226


## 6. 計算結果の保存（キャッシュ）


`detour_path_nodes`はノードIDのリストですが、GeoPackage(GPKG)はリスト型の列を直接保存できないため、JSON文字列に変換してから保存します。


In [ ]:
output_dir = "../data/interim"
os.makedirs(output_dir, exist_ok=True)

save_gdf = bridges_gdf.drop(columns=["osm_geometry_line", "osm_geometry_point"]).copy()
save_gdf["detour_path_nodes"] = save_gdf["detour_path_nodes"].apply(
    lambda nodes: json.dumps(nodes) if nodes is not None else None
)

output_path = os.path.join(output_dir, "02_bridges_with_detour.gpkg")
save_gdf.to_file(output_path, driver="GPKG")
print(f"saved: {output_path} ({len(save_gdf)} rows)")


saved: ../data/interim/02_bridges_with_detour.gpkg (1226 rows)


## 7. 迂回路ルートのジオメトリ保存 — ダッシュボードでの経路表示用

**このセクションを追加した理由：** ダッシュボード（Streamlitアプリ）で「迂回路を地図上に表示するオン/オフ」を実現するには、`detour_path_nodes`（OSMノードIDの並び）だけでなく、実際の経路の**線としての座標データ**が必要です。ノードIDだけではアプリ側でOSM道路網（`.graphml`、サイズが大きくGitHubには含めていません）を持たないと線を描けないためです。

**⚠️ このセクションは独立して再実行できます：** セクション5・6の実行結果（`bridges_gdf`・`G`）に依存せず、直前のセクション6で**保存済みの`02_bridges_with_detour.gpkg`をファイルから読み込み直す**設計にしています。そのため、Colabを新しく開き直した場合でも、重い最短経路探索（約1時間）をやり直す必要はありません。必要なのは、①保存済みの`02_bridges_with_detour.gpkg`と、②`tokyo_23_wards_drive.graphml`の2つのファイルだけです（どちらも既に手元にあるはずのファイルで、新規計算は発生しません）。


In [ ]:
import os

import geopandas as gpd
import osmnx as ox

# 保存済みの迂回路計算結果を読み込む（セクション5・6の実行直後でなくてもOK）
DETOUR_CACHE_PATH = "/content/02_bridges_with_detour.gpkg"
# ノートブックをリポジトリ構成のまま実行する場合は、以下に切り替えてください
# DETOUR_CACHE_PATH = "../data/interim/02_bridges_with_detour.gpkg"

if not os.path.exists(DETOUR_CACHE_PATH):
    raise FileNotFoundError(
        f"{DETOUR_CACHE_PATH} が見つかりません。"
        " 先にセクション5・6を実行し、02_bridges_with_detour.gpkg を作成してください。"
    )

route_source_gdf = gpd.read_file(DETOUR_CACHE_PATH)
print(f"読み込んだ橋梁数: {len(route_source_gdf)}")


読み込んだ橋梁数: 1226


In [ ]:
# OSM道路ネットワークを読み込む（既にメモリ上のGがあれば再利用し、なければファイルから読み込む。
# いずれも「読み込み」だけであり、1件ごとの最短経路探索（重い処理）は行わない）
if "G" not in dir():
    OSM_GRAPHML_PATH = "/content/tokyo_23_wards_drive.graphml"
    # OSM_GRAPHML_PATH = "../data/raw/tokyo_23_wards_drive.graphml"  # リポジトリ構成で実行する場合
    if not os.path.exists(OSM_GRAPHML_PATH):
        raise FileNotFoundError(
            f"{OSM_GRAPHML_PATH} が見つかりません。"
            " 01_data_preparation.ipynb で作成した道路ネットワークファイルを配置してください。"
        )
    G = ox.load_graphml(filepath=OSM_GRAPHML_PATH)
    print("道路ネットワークを読み込みました。")
else:
    print("既にメモリ上にある道路ネットワーク(G)を再利用します。")


道路ネットワークを読み込みました。


In [ ]:
import json

from osmnx.routing import route_to_gdf
from shapely.ops import linemerge


def build_detour_route_wkt(G, path_nodes_json):
    """保存済みの迂回路ノードID列(JSON文字列)から、経路のジオメトリをWKT文字列として復元する。

    最短経路探索（重い処理）は行わず、既に確定した経路(path_nodes)を
    実際の道路edgeの座標列に変換するだけの軽い処理。

    Args:
        G: OSMの道路ネットワークグラフ
        path_nodes_json: 迂回路のノードID列（JSON文字列。迂回路がなかった場合はNone）

    Returns:
        経路を表すLineStringのWKT文字列。迂回路がない場合はNone。
    """
    if path_nodes_json is None:
        return None
    path_nodes = json.loads(path_nodes_json)
    if path_nodes is None or len(path_nodes) < 2:
        return None
    edges_gdf = route_to_gdf(G, path_nodes, weight="length")
    merged_line = linemerge(list(edges_gdf.geometry))
    return merged_line.wkt


route_source_gdf["detour_route_wkt"] = route_source_gdf["detour_path_nodes"].apply(
    lambda nodes_json: build_detour_route_wkt(G, nodes_json)
)

route_count = route_source_gdf["detour_route_wkt"].notna().sum()
print(f"経路ジオメトリを復元できた橋梁数: {route_count} / {len(route_source_gdf)}")


経路ジオメトリを復元できた橋梁数: 1105 / 1226


`detour_route_wkt`列を追加した状態で、同じファイルに上書き保存します。ノードの座標系はOSMグラフ本来の座標系（緯度経度、EPSG:4326）のままなので、Streamlitアプリ側で追加の座標変換をせずそのまま地図表示に利用できます。


In [ ]:
route_source_gdf.to_file(DETOUR_CACHE_PATH, driver="GPKG")
print(f"saved (upserted detour_route_wkt): {DETOUR_CACHE_PATH} ({len(route_source_gdf)} rows)")


saved (upserted detour_route_wkt): /content/02_bridges_with_detour.gpkg (1226 rows)


## 8. サンプル可視化：迂回路が計算できたか地図で確認


In [ ]:
from shapely import wkt as shapely_wkt

sample_bridges = route_source_gdf[route_source_gdf["detour_route_wkt"].notna()].head(5)

center = sample_bridges.to_crs(epsg=4326).geometry.iloc[0]
m = folium.Map(location=[center.y, center.x], zoom_start=13)

for _, row in sample_bridges.iterrows():
    dpf_point = gpd.GeoSeries([row.geometry], crs=route_source_gdf.crs).to_crs(epsg=4326).iloc[0]
    folium.CircleMarker(
        [dpf_point.y, dpf_point.x], radius=5, color="blue", fill=True, tooltip=row["bridge_name"]
    ).add_to(m)

    # セクション7で作成済みのdetour_route_wkt(緯度経度)をそのまま使う。
    # 最短経路探索やG.graph["crs"]への依存はなく、再計算は発生しない。
    detour_line = shapely_wkt.loads(row["detour_route_wkt"])
    folium.GeoJson(
        detour_line,
        style_function=lambda x: {"color": "green", "weight": 3, "opacity": 0.7},
        tooltip=f"迂回路長: {row['detour_length_m']:.0f}m",
    ).add_to(m)

m


## まとめ

- 全1,226件の橋梁について、通行止め時の迂回路長を計算し、`../data/interim/02_bridges_with_detour.gpkg` として保存しました。
- あわせて、ダッシュボードでの経路表示用に迂回路のジオメトリ（`detour_route_wkt`）も保存しています。
- 次は `03_eda_clustering.ipynb` で、迂回路が見つからなかった橋（`inf`）の扱いを含めたEDA・クラスタリングを行います。
